In [33]:
import sys
import pandas as pd

import univariate_tools as ut

<font color = 'gold'> CARGAR DATOS

In [34]:
# df = pd.read_csv("Credit_score_cleaned_data.csv")

data = pd.read_csv("application_record.csv")

record = pd.read_csv("credit_record.csv")

<font color = 'gold'> CONCATENAR DATOS

In [35]:
# Obtener el mes más antiguo (mínimo) para cada ID
begin_month = record.groupby("ID")[["MONTHS_BALANCE"]].agg("min").reset_index()

# Renombrar la columna para mayor claridad sobre la variable
begin_month = begin_month.rename(columns={'MONTHS_BALANCE': 'begin_month'})

# Unir los datos originales con la nueva columna usando "left join"
new_data = pd.merge(data, begin_month, how="left", on="ID")

new_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438557 entries, 0 to 438556
Data columns (total 19 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   438557 non-null  int64  
 1   CODE_GENDER          438557 non-null  object 
 2   FLAG_OWN_CAR         438557 non-null  object 
 3   FLAG_OWN_REALTY      438557 non-null  object 
 4   CNT_CHILDREN         438557 non-null  int64  
 5   AMT_INCOME_TOTAL     438557 non-null  float64
 6   NAME_INCOME_TYPE     438557 non-null  object 
 7   NAME_EDUCATION_TYPE  438557 non-null  object 
 8   NAME_FAMILY_STATUS   438557 non-null  object 
 9   NAME_HOUSING_TYPE    438557 non-null  object 
 10  DAYS_BIRTH           438557 non-null  int64  
 11  DAYS_EMPLOYED        438557 non-null  int64  
 12  FLAG_MOBIL           438557 non-null  int64  
 13  FLAG_WORK_PHONE      438557 non-null  int64  
 14  FLAG_PHONE           438557 non-null  int64  
 15  FLAG_EMAIL       

<font color = 'gold'> VARIABLE DEPENDIENTE

In [36]:
ut.show_unique_values(record, 'STATUS')

,,,,,,,,
,X,0,C,1,2,3,4,5


In [37]:
# Inicializar la columna con valores None
record['dep_value'] = None

# Asignar "Yes" cuando STATUS sea '2', '3', '4' o '5'
record.loc[record['STATUS'].isin(['2', '3', '4', '5']), 'dep_value'] = 'Yes'

# Contar valores por ID y seleccionar la columna 'dep_value'
cpunt = record.groupby('ID').count()[['dep_value']]

# Asignar valores categóricos correctamente
cpunt['dep_value'] = cpunt['dep_value'].apply(lambda x: 'Yes' if x > 0 else 'No')

# Hacer el merge con new_data
new_data = pd.merge(new_data, cpunt, how='inner', on='ID')

# Crear la variable target con valores binarios
new_data['target'] = new_data['dep_value'].map({'Yes': 1, 'No': 0})

new_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   36457 non-null  int64  
 1   CODE_GENDER          36457 non-null  object 
 2   FLAG_OWN_CAR         36457 non-null  object 
 3   FLAG_OWN_REALTY      36457 non-null  object 
 4   CNT_CHILDREN         36457 non-null  int64  
 5   AMT_INCOME_TOTAL     36457 non-null  float64
 6   NAME_INCOME_TYPE     36457 non-null  object 
 7   NAME_EDUCATION_TYPE  36457 non-null  object 
 8   NAME_FAMILY_STATUS   36457 non-null  object 
 9   NAME_HOUSING_TYPE    36457 non-null  object 
 10  DAYS_BIRTH           36457 non-null  int64  
 11  DAYS_EMPLOYED        36457 non-null  int64  
 12  FLAG_MOBIL           36457 non-null  int64  
 13  FLAG_WORK_PHONE      36457 non-null  int64  
 14  FLAG_PHONE           36457 non-null  int64  
 15  FLAG_EMAIL           36457 non-null 

In [38]:
print(cpunt['dep_value'].value_counts())
cpunt['dep_value'].value_counts(normalize=True)

dep_value
No     45318
Yes      667
Name: count, dtype: int64


dep_value
No     0.985495
Yes    0.014505
Name: proportion, dtype: float64

<font color = 'gold'> ELIMINAR NULOS

In [39]:
new_data = new_data.mask(new_data == 'NULL').dropna()

In [40]:
new_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25134 entries, 2 to 36456
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   25134 non-null  int64  
 1   CODE_GENDER          25134 non-null  object 
 2   FLAG_OWN_CAR         25134 non-null  object 
 3   FLAG_OWN_REALTY      25134 non-null  object 
 4   CNT_CHILDREN         25134 non-null  int64  
 5   AMT_INCOME_TOTAL     25134 non-null  float64
 6   NAME_INCOME_TYPE     25134 non-null  object 
 7   NAME_EDUCATION_TYPE  25134 non-null  object 
 8   NAME_FAMILY_STATUS   25134 non-null  object 
 9   NAME_HOUSING_TYPE    25134 non-null  object 
 10  DAYS_BIRTH           25134 non-null  int64  
 11  DAYS_EMPLOYED        25134 non-null  int64  
 12  FLAG_MOBIL           25134 non-null  int64  
 13  FLAG_WORK_PHONE      25134 non-null  int64  
 14  FLAG_PHONE           25134 non-null  int64  
 15  FLAG_EMAIL           25134 non-null  int6

<font color = 'gold'> EXPORTAR DATASET

In [47]:
new_data.to_csv('dataset.csv', index=False)

#A = pd.read_csv('dataset.csv')